In [2]:
import os, pathlib, logging
from dotenv import load_dotenv

# Load all variables from .env into the environment
load_dotenv()

True

In [4]:

from config import INDUSTRY, DB_AVAILABLE, logger
from src.query_runner  import SQLQueryRunner
from src.data_extractor import DataExtractor


def main() -> None:
    logger.info("=" * 60)
    logger.info(f"  MODULE 03 — SQL AND POSTGRESQL")
    logger.info(f"  Industry: {INDUSTRY}")
    logger.info(f"  DB Available: {DB_AVAILABLE}")
    logger.info("=" * 60)

    # ── PART 1: SQL Demonstrations ──────────────────────────────────
    # These run during the teaching session to show SQL concepts live.
    runner = SQLQueryRunner()

    print("\n── DEMO 1: Basic SELECT and WHERE")
    runner.demo_basics()

    print("\n── DEMO 2: GROUP BY and Aggregation")
    runner.demo_aggregation()

    print("\n── DEMO 3: JOIN — employees with sales")
    runner.demo_joins()

    # ── PART 2: Production Extraction ──────────────────────────────
    # This is the REAL deliverable — produces raw-data.csv for Module 05
    logger.info("\n[EXTRACT] Starting production extraction...")
    extractor = DataExtractor()
    extractor.extract().save().report()


if __name__ == "__main__":
    main()



2026-04-30 14:11:08 [INFO] ============================================================
2026-04-30 14:11:08 [INFO]   MODULE 03 — SQL AND POSTGRESQL
2026-04-30 14:11:08 [INFO]   Industry: healthcare
2026-04-30 14:11:08 [INFO]   DB Available: True
2026-04-30 14:11:08 [INFO] ============================================================
2026-04-30 14:11:08 [INFO] SQLQueryRunner ready — db_available: True



── DEMO 1: Basic SELECT and WHERE

── DISTINCT departments:


2026-04-30 14:11:08 [ERROR] [SQL] Query failed: (psycopg2.errors.UndefinedTable) relation "healthcare.employees" does not exist
LINE 1: SELECT DISTINCT department FROM healthcare.employees ORDER B...
                                        ^

[SQL: SELECT DISTINCT department FROM healthcare.employees ORDER BY department]
(Background on this error at: https://sqlalche.me/e/20/f405)



── Employees earning over £100k:


2026-04-30 14:11:09 [ERROR] [SQL] Query failed: (psycopg2.errors.UndefinedTable) relation "healthcare.employees" does not exist
LINE 1: ...CT first_name, last_name, department, salary FROM healthcare...
                                                             ^

[SQL: SELECT first_name, last_name, department, salary FROM healthcare.employees WHERE salary > 100000 ORDER BY salary DESC LIMIT 10]
(Background on this error at: https://sqlalche.me/e/20/f405)


...


── DEMO 2: GROUP BY and Aggregation

── Department Salary Summary:


2026-04-30 14:11:09 [ERROR] [SQL] Query failed: (psycopg2.errors.UndefinedTable) relation "healthcare.employees" does not exist
LINE 7:             FROM healthcare.employees
                         ^

[SQL: 
            SELECT
                department,
                COUNT(*) AS headcount,
                ROUND(AVG(salary)::NUMERIC, 0) AS avg_salary,
                ROUND(MAX(salary)::NUMERIC, 0) AS max_salary
            FROM healthcare.employees
            WHERE salary IS NOT NULL
            GROUP BY department
            ORDER BY avg_salary DESC
        ]
(Background on this error at: https://sqlalche.me/e/20/f405)



── DEMO 3: JOIN — employees with sales

── Top 10 Employees by Sales Revenue:


2026-04-30 14:11:10 [ERROR] [SQL] Query failed: (psycopg2.errors.UndefinedTable) relation "healthcare.employees" does not exist
LINE 6:             FROM healthcare.employees AS e
                         ^

[SQL: 
            SELECT
                e.first_name, e.last_name, e.department,
                COUNT(s.sale_id) AS num_sales,
                ROUND(COALESCE(SUM(s.total_amount), 0)::NUMERIC, 2) AS total_revenue
            FROM healthcare.employees AS e
            LEFT JOIN healthcare.sales AS s ON e.employee_id = s.employee_id
            WHERE e.is_active = TRUE
            GROUP BY e.employee_id, e.first_name, e.last_name, e.department
            ORDER BY total_revenue DESC
            LIMIT 10
        ]
(Background on this error at: https://sqlalche.me/e/20/f405)
2026-04-30 14:11:10 [INFO] 
[EXTRACT] Starting production extraction...
2026-04-30 14:11:10 [INFO] SQLQueryRunner ready — db_available: True
2026-04-30 14:11:10 [INFO] [EXTRACT] Starting extraction — industry: hea


  MODULE 03 — EXTRACTION COMPLETE | HEALTHCARE
  Rows extracted:    300
  Columns:           17
  Output file:       raw-data.csv
  File size:         39.5 KB

  DATA QUALITY ISSUES IN RAW DATA (intentional — Module 05 will fix):
    NULL salary: 7 rows (2.3%)
    NULL performance_rating: 118 rows (39.3%)
    NULL city: 45 rows (15.0%)
    Negative salaries: 2 rows

  NEXT STEP: Copy raw-data.csv to Module 05 and run:
    python module-05-data-engineering-and-etl/run.py


In [ ]:

SELECT
    p.patient_id,
    p.first_name,
    p.last_name,
    p.email,
    p.phone,
    p.date_of_birth,
    p.gender,
    p.city,
    p.insurance_type,
    p.registered_at,
    -- These come from the sales table via the aggregated subquery below
    pat_billing.num_bills,   
    pat_billing.lifetime_billed,
    pat_billing.paid_by_patient,
    pat_billing.paid_by_insurance,
    pat_billing.recovery_rate_pct,
    pat_billing.outstanding_balance,
    pat_billing.last_claim_date,
    '{industry}'::VARCHAR AS source_schema,
    CURRENT_DATE          AS extracted_date
FROM {industry}.patients AS p
LEFT JOIN (
    SELECT
        patient_id,
        COUNT(*) AS num_bills,
    	ROUND(SUM(amount_charged)::NUMERIC, 2) AS lifetime_billed,
    	ROUND(SUM(patient_paid)::NUMERIC, 2) AS paid_by_patient,
    	ROUND(SUM(insurance_paid)::NUMERIC, 2) AS paid_by_insurance,
    	ROUND(
        	(SUM(patient_paid) + SUM(insurance_paid)) * 100.0 / NULLIF(SUM(amount_charged), 0),
        	1
    	) AS recovery_rate_pct,
    	ROUND(
        SUM(amount_charged - patient_paid - insurance_paid)::NUMERIC, 2
    	) AS outstanding_balance,
    	MAX(bill_date) AS last_claim_date
    FROM healthcare.billing
    GROUP BY patient_id
) AS pat_billing
    ON p.patient_id = pat_billing.patient_id
ORDER BY outstanding_balance DESC NULLS LAST;


NameError: name '__file__' is not defined

In [13]:
import pathlib
import os

try:
    # Works when run as a script or via pytest
    PROJECT_ROOT = pathlib.Path(__file__).resolve().parents[1]  # ↑ goes from tests/ → project root
except NameError:
    # Fallback for Jupyter/REPL
    PROJECT_ROOT = pathlib.Path(os.getcwd())

DATA_DIR = PROJECT_ROOT / "data"
SQL_DIR  = PROJECT_ROOT / "sql"

# Create folders if missing (optional but safe)
DATA_DIR.mkdir(exist_ok=True)
SQL_DIR.mkdir(exist_ok=True)

# 🔍 DEBUG: Verify paths before running assertions
print(f"PROJECT_ROOT → {PROJECT_ROOT}")
print(f"SQL_DIR      → {SQL_DIR}")
print(f"01_basics.sql exists? → {(SQL_DIR / 'sql_data_extractor.sql').exists()}")

PROJECT_ROOT → /Users/ai_engineering/healthcare_pipeline
SQL_DIR      → /Users/ai_engineering/healthcare_pipeline/sql
01_basics.sql exists? → True


In [14]:
def test_sql_files_exist():
    """All 5 SQL teaching files must exist in the sql/ directory."""
    from config import SQL_DIR
    expected = ["01_basics.sql","02_aggregation.sql","03_joins.sql",
                "04_advanced.sql","sql_data_extractor.sql"]
    for fname in expected:
        assert (SQL_DIR / fname).exists(), f"SQL file missing: {fname}"
    print("  PASS: test_sql_files_exist")

In [27]:
# ================================================================
# tests/test_sql.py — Unit Tests for Module 03
# ================================================================

import sys, pathlib
_root = pathlib.Path(__file__).resolve().parent.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import pandas as pd
from src.query_runner   import SQLQueryRunner
from src.data_extractor import DataExtractor


def test_sql_files_exist():
    """All 5 SQL teaching files must exist in the sql/ directory."""
    from config import SQL_DIR
    expected = ["01_basics.sql","02_aggregation.sql","03_joins.sql",
                "04_advanced.sql","sql_data_extractor.sql"]
    for fname in expected:
        assert (SQL_DIR / fname).exists(), f"SQL file missing: {fname}"
    print("  PASS: test_sql_files_exist")


def test_sql_files_contain_select_keyword():
    """Each SQL file must contain at least one SELECT statement."""
    from config import SQL_DIR
    for fname in ["01_basics.sql","02_aggregation.sql","03_joins.sql", "sql_data_extractor.sql"]:
        content = (SQL_DIR / fname).read_text()
        assert "SELECT" in content.upper(), f"No SELECT found in {fname}"
    print("  PASS: test_sql_files_contain_select_keyword")


def test_extract_sql_contains_industry_placeholder():
    """05_extract_raw_data.sql must use {industry} placeholder."""
    from config import SQL_DIR
    content = (SQL_DIR / "sql_data_extractor.sql").read_text()
    assert "{industry}" in content,         "Extraction SQL must use {industry} placeholder for multi-schema support"
    print("  PASS: test_extract_sql_contains_industry_placeholder")


def test_query_runner_returns_dataframe():
    """SQLQueryRunner.run() must always return a DataFrame (never crashes)."""
    runner = SQLQueryRunner()
    # Pass a trivially simple query — works even if industry table does not exist
    df = runner.run("SELECT 1 AS test_col")
    assert isinstance(df, pd.DataFrame), "run() must always return a DataFrame"
    print("  PASS: test_query_runner_returns_dataframe")


def test_query_runner_handles_bad_sql_gracefully():
    """A broken query should return empty DataFrame, not crash."""
    runner = SQLQueryRunner()
    df = runner.run("THIS IS NOT VALID SQL AT ALL")
    assert isinstance(df, pd.DataFrame),         "run() should return empty DataFrame on SQL error, not raise exception"
    print("  PASS: test_query_runner_handles_bad_sql_gracefully")


def test_query_runner_history_records_each_run():
    """Every query run must appear in the history log."""
    runner = SQLQueryRunner()
    initial_count = len(runner.history)
    runner.run("SELECT 1")
    runner.run("SELECT 2")
    assert len(runner.history) == initial_count + 2,         "history should record each query run"
    print("  PASS: test_query_runner_history_records_each_run")


def test_extractor_synthetic_data_has_required_columns():
    """Synthetic fallback data must contain all required columns."""
    raw = DataExtractor._synthetic_raw_data(50)
    required = ["employee_id","department","salary","performance_rating",
                "years_experience","source_schema","extracted_date"]
    for col in required:
        assert col in raw.columns, f"Synthetic data missing column: {col}."
    print("  PASS: test_extractor_synthetic_data_has_required_columns")


def test_extractor_synthetic_data_has_quality_issues():
    """Synthetic data must contain intentional data quality issues."""
    raw = DataExtractor._synthetic_raw_data(300)
    # Should have some NULL salaries
    null_salaries = raw["salary"].isna().sum()
    assert null_salaries > 0,         "Synthetic raw data should have some NULL salaries (data quality issues)"
    # Should have some 99-experience entries
    weird_exp = (raw["years_experience"] == 99).sum()
    assert weird_exp > 0,         "Synthetic raw data should have some years_experience=99 entries"
    print("  PASS: test_extractor_synthetic_data_has_quality_issues")


def test_extractor_save_creates_csv(tmp_path):
    """DataExtractor.save() must create raw-data.csv."""
    import pathlib
    import config as cfg
    orig = cfg.RAW_DATA_PATH
    cfg.RAW_DATA_PATH = pathlib.Path(tmp_path) / "raw-data.csv"

    extractor = DataExtractor()
    extractor.raw_df  = DataExtractor._synthetic_raw_data(50)
    extractor._status = "extracted"
    extractor.save()

    assert cfg.RAW_DATA_PATH.exists(), "save() should create raw-data.csv"
    reloaded = pd.read_csv(cfg.RAW_DATA_PATH)
    assert len(reloaded) == 50, "Saved CSV should have correct row count"
    cfg.RAW_DATA_PATH = orig
    print("  PASS: test_extractor_save_creates_csv")


if __name__ == "__main__":
    print()
    print("=" * 60)
    print("  MODULE 03 — SQL UNIT TESTS")
    print("=" * 60)
    print()
    import tempfile, pathlib

    test_sql_files_exist()
    test_sql_files_contain_select_keyword()
    test_extract_sql_contains_industry_placeholder()
    test_query_runner_returns_dataframe()
    test_query_runner_handles_bad_sql_gracefully()
    test_query_runner_history_records_each_run()
    test_extractor_synthetic_data_has_required_columns()
    test_extractor_synthetic_data_has_quality_issues()

    with tempfile.TemporaryDirectory() as tmp:
        test_extractor_save_creates_csv(pathlib.Path(tmp))

    print()
    print("=" * 60)
    print("  All tests passed ✓")
    print("=" * 60)


NameError: name '__file__' is not defined

In [28]:
# config.py
import pathlib
ROOT_DIR = pathlib.Path(__file__).resolve().parent
SQL_DIR = ROOT_DIR / "sql"


NameError: name '__file__' is not defined

In [29]:
# Use .cwd() because __file__ is only for .py scripts
ROOT_DIR = pathlib.Path(os.getcwd()).resolve()
SQL_DIR = ROOT_DIR / "sql"
RAW_DATA_PATH = ROOT_DIR / "data" / "raw-data.csv"

# Optional: Ensure the directories exist so the tests don't fail
SQL_DIR.mkdir(exist_ok=True)
(ROOT_DIR / "data").mkdir(exist_ok=True)

print(f"Paths set! SQL_DIR is: {SQL_DIR}")

Paths set! SQL_DIR is: /Users/ai_engineering/healthcare_pipeline/sql


In [30]:
def test_sql_files_exist():
    """Verify that the required extraction SQL file exists."""
    from config import SQL_DIR
    # Removed "01_basics.sql" and others; only keeping what you need
    expected = ["sql_data_extractor.sql"]
    
    for fname in expected:
        assert (SQL_DIR / fname).exists(), f"SQL file missing: {fname}"
    print("  PASS: test_sql_files_exist")


In [37]:

def test_sql_files_exist():
    """All 5 SQL teaching files must exist in the sql/ directory."""
    from config import SQL_DIR
    expected = ["sql_data_extractor.sql"]
    for fname in expected:
        assert (SQL_DIR / fname).exists(), f"SQL file missing: {fname}"
    print("  PASS: test_sql_files_exist")


def test_sql_files_contain_select_keyword():
    """Each SQL file must contain at least one SELECT statement."""
    from config import SQL_DIR
    for fname in ["sql_data_extractor.sql"]:
        content = (SQL_DIR / fname).read_text()
        assert "SELECT" in content.upper(), f"No SELECT found in {fname}"
    print("  PASS: test_sql_files_contain_select_keyword")


def test_extract_sql_contains_industry_placeholder():
    """05_extract_raw_data.sql must use {industry} placeholder."""
    from config import SQL_DIR
    content = (SQL_DIR / "sql_data_extractor.sql").read_text()
    assert "{industry}" in content,         "Extraction SQL must use {industry} placeholder for multi-schema support"
    print("  PASS: test_extract_sql_contains_industry_placeholder")


def test_query_runner_returns_dataframe():
    """SQLQueryRunner.run() must always return a DataFrame (never crashes)."""
    runner = SQLQueryRunner()
    # Pass a trivially simple query — works even if industry table does not exist
    df = runner.run("SELECT 1 AS test_col")
    assert isinstance(df, pd.DataFrame), "run() must always return a DataFrame"
    print("  PASS: test_query_runner_returns_dataframe")


def test_query_runner_handles_bad_sql_gracefully():
    """A broken query should return empty DataFrame, not crash."""
    runner = SQLQueryRunner()
    df = runner.run("THIS IS NOT VALID SQL AT ALL")
    assert isinstance(df, pd.DataFrame),         "run() should return empty DataFrame on SQL error, not raise exception"
    print("  PASS: test_query_runner_handles_bad_sql_gracefully")


def test_query_runner_history_records_each_run():
    """Every query run must appear in the history log."""
    runner = SQLQueryRunner()
    initial_count = len(runner.history)
    runner.run("SELECT 1")
    runner.run("SELECT 2")
    assert len(runner.history) == initial_count + 2,         "history should record each query run"
    print("  PASS: test_query_runner_history_records_each_run")


In [ ]:
    test_sql_files_exist()
    test_sql_files_contain_select_keyword()
    test_extract_sql_contains_industry_placeholder()
    test_query_runner_returns_dataframe()
    test_query_runner_handles_bad_sql_gracefully()
    test_query_runner_history_records_each_run()

FileNotFoundError: [Errno 2] No such file or directory: '/Users/ai_engineering/sql/sql_data_extractor.sql'

In [6]:
import sys, pathlib

try:
    # Works when running as a .py script
    _root = pathlib.Path(__file__).resolve().parent.parent
except NameError:
    # Works when running in a Jupyter Notebook/Interactive shell
    _root = pathlib.Path.cwd().parent

if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))


import pandas as pd
from src.query_runner   import SQLQueryRunner
from src.data_extractor import DataExtractor


def test_sql_files_exist():
    """All 5 SQL teaching files must exist in the sql/ directory."""
    from config import SQL_DIR
    expected = ["sql_data_extractor.sql"]
    for fname in expected:
        assert (SQL_DIR / fname).exists(), f"SQL file missing: {fname}"
    print("  PASS: test_sql_files_exist")

In [22]:
def test_sql_files_contain_select_keyword():
    """Each SQL file must contain at least one SELECT statement."""
    from config import SQL_DIR
    for fname in ["sql_data_extractor.sql"]:
        content = (SQL_DIR / fname).read_text()
        assert "SELECT" in content.upper(), f"No SELECT found in {fname}"
    print("  PASS: test_sql_files_contain_select_keyword")


def test_extract_sql_contains_industry_placeholder():
    """05_extract_raw_data.sql must use {industry} placeholder."""
    from config import SQL_DIR
    content = (SQL_DIR / "sql_data_extractor.sql").read_text()
    assert "{industry}" in content,         "Extraction SQL must use {industry} placeholder for multi-schema support"
    print("  PASS: test_extract_sql_contains_industry_placeholder")



In [14]:
test_sql_files_exist()

  PASS: test_sql_files_exist


In [20]:
test_sql_files_contain_select_keyword()

  PASS: test_sql_files_contain_select_keyword


In [23]:
test_extract_sql_contains_industry_placeholder()

  PASS: test_extract_sql_contains_industry_placeholder


In [24]:
def test_query_runner_returns_dataframe():
    """SQLQueryRunner.run() must always return a DataFrame (never crashes)."""
    runner = SQLQueryRunner()
    # Pass a trivially simple query — works even if industry table does not exist
    df = runner.run("SELECT 1 AS test_col")
    assert isinstance(df, pd.DataFrame), "run() must always return a DataFrame"
    print("  PASS: test_query_runner_returns_dataframe")


def test_query_runner_handles_bad_sql_gracefully():
    """A broken query should return empty DataFrame, not crash."""
    runner = SQLQueryRunner()
    df = runner.run("THIS IS NOT VALID SQL AT ALL")
    assert isinstance(df, pd.DataFrame),         "run() should return empty DataFrame on SQL error, not raise exception"
    print("  PASS: test_query_runner_handles_bad_sql_gracefully")


In [ ]:
test_sql_files_exist()
test_sql_files_contain_select_keyword()
test_extract_sql_contains_industry_placeholder()



2026-04-30 14:30:50 [INFO] SQLQueryRunner ready — db_available: True


  PASS: test_sql_files_exist
  PASS: test_sql_files_contain_select_keyword
  PASS: test_extract_sql_contains_industry_placeholder


2026-04-30 14:30:50 [INFO] [SQL] Query complete — 1 rows × 1 cols | 369.7ms
2026-04-30 14:30:50 [INFO] SQLQueryRunner ready — db_available: True


  PASS: test_query_runner_returns_dataframe


2026-04-30 14:30:50 [ERROR] [SQL] Query failed: (psycopg2.errors.SyntaxError) syntax error at or near "THIS"
LINE 1: THIS IS NOT VALID SQL AT ALL
        ^

[SQL: THIS IS NOT VALID SQL AT ALL]
(Background on this error at: https://sqlalche.me/e/20/f405)


  PASS: test_query_runner_handles_bad_sql_gracefully


In [ ]:
test_query_runner_returns_dataframe()


2026-04-30 14:33:39 [INFO] SQLQueryRunner ready — db_available: True
2026-04-30 14:33:39 [INFO] [SQL] Query complete — 1 rows × 1 cols | 403.9ms
2026-04-30 14:33:39 [INFO] SQLQueryRunner ready — db_available: True


  PASS: test_query_runner_returns_dataframe


2026-04-30 14:33:39 [ERROR] [SQL] Query failed: (psycopg2.errors.SyntaxError) syntax error at or near "THIS"
LINE 1: THIS IS NOT VALID SQL AT ALL
        ^

[SQL: THIS IS NOT VALID SQL AT ALL]
(Background on this error at: https://sqlalche.me/e/20/f405)


  PASS: test_query_runner_handles_bad_sql_gracefully


In [31]:
test_query_runner_handles_bad_sql_gracefully()

2026-04-30 14:33:52 [INFO] SQLQueryRunner ready — db_available: True
2026-04-30 14:33:52 [ERROR] [SQL] Query failed: (psycopg2.errors.SyntaxError) syntax error at or near "THIS"
LINE 1: THIS IS NOT VALID SQL AT ALL
        ^

[SQL: THIS IS NOT VALID SQL AT ALL]
(Background on this error at: https://sqlalche.me/e/20/f405)


  PASS: test_query_runner_handles_bad_sql_gracefully


In [ ]:

def test_query_runner_history_records_each_run():
    """Every query run must appear in the history log."""
    runner = SQLQueryRunner()
    initial_count = len(runner.history)
    runner.run("SELECT 1")
    runner.run("SELECT 2")
    assert len(runner.history) == initial_count + 2,         "history should record each query run"
    print("  PASS: test_query_runner_history_records_each_run")


def test_extractor_synthetic_data_has_required_columns():
    """Synthetic fallback data must contain all required columns."""
    raw = DataExtractor._synthetic_raw_data(50)
    required = ["employee_id","department","salary","performance_rating",
                "years_experience","source_schema","extracted_date"]
    for col in required:
        assert col in raw.columns, f"Synthetic data missing column: {col}."
    print("  PASS: test_extractor_synthetic_data_has_required_columns")


def test_extractor_synthetic_data_has_quality_issues():
    """Synthetic data must contain intentional data quality issues."""
    raw = DataExtractor._synthetic_raw_data(300)
    # Should have some NULL salaries
    null_salaries = raw["salary"].isna().sum()
    assert null_salaries > 0,         "Synthetic raw data should have some NULL salaries (data quality issues)"
    # Should have some 99-experience entries
    weird_exp = (raw["years_experience"] == 99).sum()
    assert weird_exp > 0,         "Synthetic raw data should have some years_experience=99 entries"
    print("  PASS: test_extractor_synthetic_data_has_quality_issues")


def test_extractor_save_creates_csv(tmp_path):
    """DataExtractor.save() must create raw-data.csv."""
    import pathlib
    import config as cfg
    orig = cfg.RAW_DATA_PATH
    cfg.RAW_DATA_PATH = pathlib.Path(tmp_path) / "raw-data.csv"

    extractor = DataExtractor()
    extractor.raw_df  = DataExtractor._synthetic_raw_data(50)
    extractor._status = "extracted"
    extractor.save()

    assert cfg.RAW_DATA_PATH.exists(), "save() should create raw-data.csv"
    reloaded = pd.read_csv(cfg.RAW_DATA_PATH)
    assert len(reloaded) == 50, "Saved CSV should have correct row count"
    cfg.RAW_DATA_PATH = orig
    print("  PASS: test_extractor_save_creates_csv")


In [27]:
test_query_runner_history_records_each_run()
test_extractor_synthetic_data_has_required_columns()
test_extractor_synthetic_data_has_quality_issues()

NameError: name 'test_query_runner_history_records_each_run' is not defined

In [28]:
def test_query_runner_history_records_each_run():
    """Every query run must appear in the history log."""
    runner = SQLQueryRunner()
    initial_count = len(runner.history)
    runner.run("SELECT 1")
    runner.run("SELECT 2")
    assert len(runner.history) == initial_count + 2,         "history should record each query run"
    print("  PASS: test_query_runner_history_records_each_run")


In [29]:
test_query_runner_history_records_each_run()

2026-04-30 14:32:47 [INFO] SQLQueryRunner ready — db_available: True
2026-04-30 14:32:47 [INFO] [SQL] Query complete — 1 rows × 1 cols | 392.4ms
2026-04-30 14:32:48 [INFO] [SQL] Query complete — 1 rows × 1 cols | 368.8ms


  PASS: test_query_runner_history_records_each_run


In [34]:
test_extract_sql_contains_industry_placeholder()

  PASS: test_extract_sql_contains_industry_placeholder


In [35]:
def test_query_runner_returns_dataframe():
    """SQLQueryRunner.run() must always return a DataFrame (never crashes)."""
    runner = SQLQueryRunner()
    # Pass a trivially simple query — works even if industry table does not exist
    df = runner.run("SELECT 1 AS test_col")
    assert isinstance(df, pd.DataFrame), "run() must always return a DataFrame"
    print("  PASS: test_query_runner_returns_dataframe")

In [36]:
test_query_runner_returns_dataframe()

2026-04-30 14:39:30 [INFO] SQLQueryRunner ready — db_available: True
2026-04-30 14:39:31 [INFO] [SQL] Query complete — 1 rows × 1 cols | 379.3ms


  PASS: test_query_runner_returns_dataframe


In [32]:
def test_extractor_synthetic_data_has_required_columns():
    """Synthetic fallback data must contain all required columns."""
    raw = DataExtractor._synthetic_raw_data(50)
    required = ["patient_id","first_name","last_name","date_of_birth",
                "insurance_type","source_schema","extracted_date"]
    for col in required:
        assert col in raw.columns, f"Synthetic data missing column: {col}."
    print("  PASS: test_extractor_synthetic_data_has_required_columns")


In [33]:
test_extractor_synthetic_data_has_required_columns()

AssertionError: Synthetic data missing column: patient_id.